# PPO: Donkey Kong (Default Hyperparameters)

**Algorithm:** PPO  
**Environment:** Donkey Kong (`ALE/DonkeyKong-v5`)  
**Learning rate:** default (3e-4)  
**Parallel envs:** 8  

In [1]:
# Configuration
ALGO            = "ppo"
ENV_KEY         = "donkeykong"
TOTAL_TIMESTEPS = 2_000_000
CHECKPOINT_FREQ = 100_000
LEARNING_RATE   = None
RUN_NAME        = None

In [2]:
import os

import ale_py
import gymnasium as gym
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

gym.register_envs(ale_py)

ENV_IDS = {
    "qbert":      "QbertNoFrameskip-v4",
    "donkeykong": "ALE/DonkeyKong-v5",
}

run_id = f"{ALGO}_{ENV_KEY}" + (f"_{RUN_NAME}" if RUN_NAME else "")
checkpoint_dir = os.path.join("checkpoints", run_id)
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs("logs", exist_ok=True)

print(f"Run ID      : {run_id}")
print(f"Checkpoints : {checkpoint_dir}")

Run ID      : ppo_donkeykong
Checkpoints : checkpoints/ppo_donkeykong


In [3]:
n_envs = 8 if ALGO == "ppo" else 1
vec_env = make_atari_env(ENV_IDS[ENV_KEY], n_envs=n_envs, seed=0)
vec_env = VecFrameStack(vec_env, n_stack=4)
print(f"Environment : {ENV_IDS[ENV_KEY]}  |  n_envs: {n_envs}")

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Environment : ALE/DonkeyKong-v5  |  n_envs: 8


## Fresh Training

Checkpoints saved automatically to `checkpoints/ppo_donkeykong/` every 100,000 steps.

In [4]:
if ALGO == "dqn":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        buffer_size=100_000, learning_starts=10_000,
        batch_size=32, train_freq=4, target_update_interval=1_000,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = DQN("CnnPolicy", vec_env, **kwargs)
elif ALGO == "ppo":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        n_steps=128, batch_size=256, n_epochs=4, ent_coef=0.01, clip_range=0.1,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = PPO("CnnPolicy", vec_env, **kwargs)

print(f"Algorithm     : {ALGO.upper()}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Device        : {model.device}")

Using cpu device
Wrapping the env in a VecTransposeImage.
Algorithm     : PPO
Learning rate : None
Device        : cpu


In [5]:
checkpoint_callback = CheckpointCallback(
    save_freq=max(CHECKPOINT_FREQ // n_envs, 1),
    save_path=checkpoint_dir,
    name_prefix="checkpoint",
    save_replay_buffer=False,
    save_vecnormalize=True,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback,
    tb_log_name=run_id,
    log_interval=100,
    progress_bar=True,
    reset_num_timesteps=True,
)

final_path = os.path.join(checkpoint_dir, "final_model")
model.save(final_path)
print(f"\nTraining complete. Final model saved to: {final_path}.zip")

Logging to logs/ppo_donkeykong_2


Output()

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 838         |
|    ep_rew_mean          | 307         |
| time/                   |             |
|    fps                  | 187         |
|    iterations           | 100         |
|    time_elapsed         | 546         |
|    total_timesteps      | 102400      |
| train/                  |             |
|    approx_kl            | 0.006805814 |
|    clip_fraction        | 0.18        |
|    clip_range           | 0.1         |
|    entropy_loss         | -2.52       |
|    explained_variance   | 0.53        |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0116      |
|    n_updates            | 396         |
|    policy_gradient_loss | -0.0199     |
|    value_loss           | 0.161       |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.61e+03    |
|    ep_rew_mean          | 2.87e+03    |
| time/                   |             |
|    fps                  | 187         |
|    iterations           | 200         |
|    time_elapsed         | 1093        |
|    total_timesteps      | 204800      |
| train/                  |             |
|    approx_kl            | 0.006227297 |
|    clip_fraction        | 0.186       |
|    clip_range           | 0.1         |
|    entropy_loss         | -2.28       |
|    explained_variance   | 0.871       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0336      |
|    n_updates            | 796         |
|    policy_gradient_loss | -0.0121     |
|    value_loss           | 0.27        |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 3.11e+03    |
|    ep_rew_mean          | 3.66e+03    |
| time/                   |             |
|    fps                  | 187         |
|    iterations           | 300         |
|    time_elapsed         | 1636        |
|    total_timesteps      | 307200      |
| train/                  |             |
|    approx_kl            | 0.008118065 |
|    clip_fraction        | 0.236       |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.98       |
|    explained_variance   | 0.99        |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0432     |
|    n_updates            | 1196        |
|    policy_gradient_loss | -0.0199     |
|    value_loss           | 0.0181      |
-----------------------------------------


...

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 3.26e+03    |
|    ep_rew_mean          | 4.04e+03    |
| time/                   |             |
|    fps                  | 188         |
|    iterations           | 1600        |
|    time_elapsed         | 8695        |
|    total_timesteps      | 1638400     |
| train/                  |             |
|    approx_kl            | 0.010365792 |
|    clip_fraction        | 0.204       |
|    clip_range           | 0.1         |
|    entropy_loss         | -2.02       |
|    explained_variance   | 0.992       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0633     |
|    n_updates            | 6396        |
|    policy_gradient_loss | -0.0269     |
|    value_loss           | 0.0259      |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 3.25e+03    |
|    ep_rew_mean          | 4.01e+03    |
| time/                   |             |
|    fps                  | 188         |
|    iterations           | 1700        |
|    time_elapsed         | 9235        |
|    total_timesteps      | 1740800     |
| train/                  |             |
|    approx_kl            | 0.013906218 |
|    clip_fraction        | 0.265       |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.88       |
|    explained_variance   | 0.991       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0606     |
|    n_updates            | 6796        |
|    policy_gradient_loss | -0.0263     |
|    value_loss           | 0.0196      |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 3.34e+03    |
|    ep_rew_mean          | 4.14e+03    |
| time/                   |             |
|    fps                  | 188         |
|    iterations           | 1800        |
|    time_elapsed         | 9774        |
|    total_timesteps      | 1843200     |
| train/                  |             |
|    approx_kl            | 0.009667395 |
|    clip_fraction        | 0.254       |
|    clip_range           | 0.1         |
|    entropy_loss         | -1.86       |
|    explained_variance   | 0.989       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.053      |
|    n_updates            | 7196        |
|    policy_gradient_loss | -0.0258     |
|    value_loss           | 0.0299      |
-----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 3.3e+03    |
|    ep_rew_mean          | 4.09e+03   |
| time/                   |            |
|    fps                  | 188        |
|    iterations           | 1900       |
|    time_elapsed         | 10315      |
|    total_timesteps      | 1945600    |
| train/                  |            |
|    approx_kl            | 0.01607994 |
|    clip_fraction        | 0.235      |
|    clip_range           | 0.1        |
|    entropy_loss         | -1.84      |
|    explained_variance   | 0.874      |
|    learning_rate        | 0.0003     |
|    loss                 | 0.0128     |
|    n_updates            | 7596       |
|    policy_gradient_loss | -0.0126    |
|    value_loss           | 0.212      |
----------------------------------------



Training complete. Final model saved to: checkpoints/ppo_donkeykong/final_model.zip
